In [9]:
import os
import joblib
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
biogas = pd.read_csv("interpolated_biogas_data_8000.csv")

biogas

In [ ]:
biogas.info()

In [ ]:
biogas.describe().T

In [ ]:
biogas.hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
#to show correlation of two columns
correlation = biogas["Biogas Yield (m³/Kg)"].corr(biogas["CH4 (%)"])
print(f"correlation between Biogas Yield and CH4 produced: {correlation:.4f}")

In [ ]:
#for correlation of two columns
plt.figure(figsize=(10,6))
plt.scatter(biogas["Biogas Yield (m³/Kg)"], biogas["CH4 (%)"], alpha=0.7)
plt.title(f"correlation: {correlation:.4f}")
plt.xlabel("Biogas Yield")
plt.ylabel("CH4")
plt.grid(True,alpha=0.3)
plt.show()

In [ ]:
# features (inputs) and targets (what we want to predict)
# think of features as clues we give the robot, an targets s the answers

FEATURES = [
    "Hydraulic Retention Time (Days)",
    "Temperature (°C)",           
    "pH"                       
]
TARGETS = [
    "Biogas Yield (m³/Kg)",          
    "CH4 (%)"
]
x = biogas[FEATURES].copy()
y = biogas[TARGETS].copy()

In [ ]:
# sPLIT INTO tRAIN AND tEST SETS (LIKE HOMEWORK VS EXAM)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

In [ ]:
# Build a preprocessing + model pipeline
# We scale (StandardScaler) pH a little but trees don't need scaling
# Still, we keep a simple, safe setup for future swaps.

numeric_features = FEATURES
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ], remainder="drop"
)

# We use a RandomForest inside a MultiOutputRegressor to predict two targets at once
regressor = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators =300,
        max_depth=None,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
)

model = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", regressor)
])

In [ ]:
# Train the model (teach the robot)
print("\nTraining the model...")
model.fit(x_train, y_train)


In [ ]:
# Evaluate on the test set (see how smart the robot is on new data)
y_pred = model.predict(x_test)

# The model predicts both Biogas and CH4 in one go; split them for metrics
biogas_true = y_test["Biogas Yield (m³/Kg)"].values      
ch4_true = y_test["CH4 (%)"].values
biogas_pred = y_pred[:,0]
ch4_pred = y_pred[:,1]

# Apply business rule to predictions: if predicted biogas <= 0, force CH4 =0
# (keeps things realistic)
ch4_pred = np.where(biogas_pred <= 0.0, 0.0, ch4_pred)
biogas_pred = np.clip(biogas_pred, 0.0, None)
ch4_pred = np.clip(ch4_pred, 0.0, None)

# Metrics: MAE (lower is better), R^2 (closer to 10 is better)
biogas_mae = mean_absolute_error(biogas_true, biogas_pred)
biogas_r2 = r2_score(biogas_true, biogas_pred)
ch4_mae = mean_absolute_error(ch4_true, ch4_pred)
ch4_r2 = r2_score(ch4_true, ch4_pred)

print("\nTest Metrics:")
print(f"Biogas Yield - MAE: {biogas_mae:.4f}, R^2: {biogas_r2:.4f}")
print(f"CH4 (PERCENT) = mae: {ch4_mae:.4f}, R^2: {ch4_r2:.4f}")


In [ ]:
# Optional: quick plots (saves as PNGs so you can view the later)




In [ ]:
# Cross-validation (extra safety check on trainingdata)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
# We'll measure R^2 for Biogas only as a quick example (multioutput CV is heavier)
cv_scores = cross_val_score(model, x_train, y_train[TARGETS],
                            cv=cv, scoring="r2")
print("\n5-Fold CV R^2 (Biogas): mean=%.3f std=%.3f" % (cv_scores.mean(), cv_scores.std()))


In [ ]:
# Says the trained model  to disk so Streamlit can use it
os.makedirs("models", exist_ok=True)
MODEL_PATH = os.path.join("models", "biogas_model.joblib")
joblib.dump(model, MODEL_PATH)
print("\nSaved trained model to: ", MODEL_PATH)

In [ ]:
# Tiny helper: make a unction to predict with rule applied
# (We'll also reuse this idea inside the Streamlit app.)

def predict_biogas_and_ch4(h, t, pH):
    """Return (Biogas, ch4) prediction with business rule enforced."""
    X_in = pd.DataFrame({
        "Hydraulic Retention Time (Days)": [h],
        "Temperature (°C)": [t],
        "pH": [pH]
    })
    y_hat = model.predict(X_in)[0]
    biogas, ch4 = float(y_hat[0]), float(y_hat[1])
    # Enforce rule: when biogas is zero or negative, CH4 must be zero
    if biogas <= 0:
        ch4 = 0.0
        biogas = max(0.0, biogas)
        ch4 = max(0.0, ch4)
        return biogas, ch4
    
# Quick demo prediction
print("\nDemo prediction for HRT=10 days, Temp=30°C, pH=6.9:")
print(predict_biogas_and_ch4(20, 40, 5.4))
